# From Grid to Wardrobe: Analyzing Formula 1’s Influence on Fashion & Merch Demand

This project explores whether Formula 1 team performance and driver popularity influence fashion and merchandise demand. Using F1 race results, Google Trends data, and fashion-related search terms, this analysis studies how sports performance connects with consumer interest in F1 apparel and streetwear.

In [ ]:
!pip install pytrends plotly streamlit -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from pytrends.request import TrendReq
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
!pip install kaggle -q

In [ ]:
!kaggle datasets download -d rohanrao/formula-1-world-championship-1950-2020

In [ ]:
!unzip formula-1-world-championship-1950-2020.zip

In [ ]:
import pandas as pd

races = pd.read_csv("races.csv")
results = pd.read_csv("results.csv")
drivers = pd.read_csv("drivers.csv")
constructors = pd.read_csv("constructors.csv")
constructor_standings = pd.read_csv("constructor_standings.csv")

In [ ]:
print(races.shape)
print(results.shape)
print(drivers.shape)
print(constructors.shape)
print(constructor_standings.shape)

In [ ]:
print("Races:", races.shape)
print("Results:", results.shape)
print("Drivers:", drivers.shape)
print("Constructors:", constructors.shape)
print("Constructor standings:", constructor_standings.shape)

In [ ]:
races.head()

In [ ]:
races_recent = races[races["year"] >= 2018][["raceId", "year", "round", "name", "date"]]

races_recent.head()

In [ ]:
results_clean = results[[
    "raceId",
    "driverId",
    "constructorId",
    "grid",
    "positionOrder",
    "points"
]].copy()

drivers_clean = drivers[[
    "driverId",
    "forename",
    "surname",
    "nationality"
]].copy()

drivers_clean["driver_name"] = drivers_clean["forename"] + " " + drivers_clean["surname"]

constructors_clean = constructors[[
    "constructorId",
    "name",
    "nationality"
]].copy()

constructors_clean = constructors_clean.rename(columns={
    "name": "team_name",
    "nationality": "team_nationality"
})

In [ ]:
f1 = results_clean.merge(races_recent, on="raceId", how="inner")

f1 = f1.merge(
    drivers_clean[["driverId", "driver_name", "nationality"]],
    on="driverId",
    how="left"
)

f1 = f1.merge(
    constructors_clean,
    on="constructorId",
    how="left"
)

f1.head()

In [ ]:
f1.shape

In [ ]:
f1["podium"] = np.where(f1["positionOrder"] <= 3, 1, 0)
f1["top_5"] = np.where(f1["positionOrder"] <= 5, 1, 0)
f1["win"] = np.where(f1["positionOrder"] == 1, 1, 0)

f1.head()

In [ ]:
team_performance = f1.groupby(["year", "team_name"]).agg(
    total_points=("points", "sum"),
    podiums=("podium", "sum"),
    wins=("win", "sum"),
    avg_finish=("positionOrder", "mean")
).reset_index()

team_performance.head()

In [ ]:
team_performance.sort_values(
    by=["year", "total_points"],
    ascending=[False, False]
).head(20)

In [ ]:
import plotly.express as px

fig = px.bar(
    team_performance[team_performance["year"] == team_performance["year"].max()],
    x="team_name",
    y="total_points",
    title="F1 Team Performance: Total Points by Team",
    labels={
        "team_name": "Team",
        "total_points": "Total Points"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=500
)

fig.show()

In [ ]:
latest_year = team_performance["year"].max()

latest_team_perf = team_performance[team_performance["year"] == latest_year].sort_values(
    by="total_points",
    ascending=True
)

fig = px.bar(
    latest_team_perf,
    x="total_points",
    y="team_name",
    orientation="h",
    text="total_points",
    title=f"F1 Team Performance — Total Points by Team ({latest_year})",
    labels={
        "team_name": "",
        "total_points": "Total Points"
    },
    color="total_points",
    color_continuous_scale="Blues"
)

fig.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside",
    marker_line_width=0,
    hovertemplate="<b>%{y}</b><br>Total Points: %{x}<extra></extra>"
)

fig.update_layout(
    height=600,
    plot_bgcolor="#171717",
    paper_bgcolor="#171717",
    font=dict(color="#F2F2F2", family="Arial"),
    title=dict(
        font=dict(size=24, color="#FFFFFF"),
        x=0.02
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#333333",
        zeroline=False,
        title_font=dict(size=14),
        tickfont=dict(color="#BDBDBD")
    ),
    yaxis=dict(
        tickfont=dict(color="#F2F2F2", size=13)
    ),
    coloraxis_showscale=False,
    margin=dict(l=120, r=60, t=80, b=40)
)

fig.show()

In [ ]:
top_team = latest_team_perf.sort_values("total_points", ascending=False).iloc[0]

print(f"Top team in {latest_year}: {top_team['team_name']} with {top_team['total_points']} points.")

In [ ]:
!pip install pytrends -q

In [ ]:
from pytrends.request import TrendReq
import pandas as pd
import time

In [ ]:
search_terms = [
    "Ferrari F1 merch",
    "McLaren F1 merch",
    "Mercedes F1 merch",
    "Red Bull F1 merch",
    "Lewis Hamilton fashion"
]

all_trends = []

for term in search_terms:
    print("Pulling:", term)

    pytrends = TrendReq(hl="en-US", tz=360)

    pytrends.build_payload(
        [term],
        timeframe="2018-01-01 2026-06-01",
        geo="US"
    )

    data = pytrends.interest_over_time()

    if "isPartial" in data.columns:
        data = data.drop(columns=["isPartial"])

    data = data.reset_index()
    data = data.rename(columns={term: "search_interest"})
    data["search_term"] = term

    all_trends.append(data)

    time.sleep(15)

In [ ]:
trends_long = pd.concat(all_trends, ignore_index=True)

trends_long.head()

In [ ]:
trends_long["year"] = trends_long["date"].dt.year

In [ ]:
def assign_team(term):
    term = term.lower()

    if "ferrari" in term:
        return "Ferrari"
    elif "mclaren" in term:
        return "McLaren"
    elif "mercedes" in term:
        return "Mercedes"
    elif "red bull" in term:
        return "Red Bull"
    else:
        return "Driver/Fashion"

trends_long["team_label"] = trends_long["search_term"].apply(assign_team)

trends_long.head()

In [ ]:
import plotly.express as px

fig = px.line(
    trends_long,
    x="date",
    y="search_interest",
    color="search_term",
    title="F1 Fashion & Merch Search Interest Over Time",
    labels={
        "date": "Date",
        "search_interest": "Search Interest",
        "search_term": "Search Term"
    }
)

fig.update_traces(line=dict(width=3))

fig.update_layout(
    height=550,
    plot_bgcolor="#171717",
    paper_bgcolor="#171717",
    font=dict(color="#F2F2F2", family="Arial"),
    title=dict(
        font=dict(size=24, color="#FFFFFF"),
        x=0.02
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#333333",
        tickfont=dict(color="#BDBDBD")
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#333333",
        tickfont=dict(color="#BDBDBD")
    ),
    legend=dict(
        title="",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),
    margin=dict(l=60, r=40, t=100, b=50)
)

fig.show()

In [ ]:
fashion_demand = trends_long.groupby(["year", "team_label"]).agg(
    avg_search_interest=("search_interest", "mean"),
    max_search_interest=("search_interest", "max")
).reset_index()

fashion_demand.head()

In [ ]:
def simplify_team(name):
    name = name.lower()

    if "ferrari" in name:
        return "Ferrari"
    elif "mclaren" in name:
        return "McLaren"
    elif "mercedes" in name:
        return "Mercedes"
    elif "red bull" in name:
        return "Red Bull"
    else:
        return name.title()

team_performance["team_label"] = team_performance["team_name"].apply(simplify_team)

team_performance.head()

In [ ]:
team_performance["team_label"].unique()

In [ ]:
analysis_df = team_performance.merge(
    fashion_demand,
    on=["year", "team_label"],
    how="inner"
)

analysis_df.head()

In [ ]:
analysis_df.shape

In [ ]:
fig = px.scatter(
    analysis_df,
    x="total_points",
    y="avg_search_interest",
    color="team_label",
    size="podiums",
    hover_data=["year", "wins", "avg_finish"],
    title="F1 Performance vs Fashion/Merch Search Interest",
    labels={
        "total_points": "Total Team Points",
        "avg_search_interest": "Average Fashion/Merch Search Interest",
        "team_label": "Team",
        "podiums": "Podiums"
    }
)

fig.update_traces(
    marker=dict(
        line=dict(width=1, color="#FFFFFF"),
        opacity=0.85
    )
)

fig.update_layout(
    height=600,
    plot_bgcolor="#171717",
    paper_bgcolor="#171717",
    font=dict(color="#F2F2F2", family="Arial"),
    title=dict(
        font=dict(size=24, color="#FFFFFF"),
        x=0.02
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="#333333",
        zeroline=False,
        tickfont=dict(color="#BDBDBD")
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#333333",
        zeroline=False,
        tickfont=dict(color="#BDBDBD")
    ),
    legend=dict(
        title="Team",
        font=dict(color="#F2F2F2")
    ),
    margin=dict(l=70, r=40, t=90, b=60)
)

fig.show()

In [ ]:
correlation = analysis_df[["total_points", "podiums", "wins", "avg_finish", "avg_search_interest"]].corr()

correlation

In [ ]:
analysis_df["total_points"].corr(analysis_df["avg_search_interest"])

In [ ]:
analysis_df.to_csv("f1_fashion_analysis.csv", index=False)
trends_long.to_csv("f1_fashion_trends.csv", index=False)
team_performance.to_csv("f1_team_performance.csv", index=False)

In [ ]:
analysis_df.head()

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="F1 FashionTech Dashboard",
    page_icon="🏎️",
    layout="wide"
)

# -----------------------------
# Load Data
# -----------------------------
analysis_df = pd.read_csv("f1_fashion_analysis.csv")
trends_long = pd.read_csv("f1_fashion_trends.csv")
team_performance = pd.read_csv("f1_team_performance.csv")

trends_long["date"] = pd.to_datetime(trends_long["date"])

# -----------------------------
# CSS Styling
# -----------------------------
st.markdown("""
<style>
.stApp {
    background-color: #171717;
    color: #F2F2F2;
}

.main-title {
    font-size: 46px;
    font-weight: 800;
    color: #FFFFFF;
    margin-bottom: 0px;
}

.subtitle {
    font-size: 17px;
    color: #A8A8A8;
    margin-bottom: 30px;
}

.metric-card {
    background-color: #242423;
    padding: 25px;
    border-radius: 16px;
    border: 1px solid #343434;
    box-shadow: 0px 4px 25px rgba(0,0,0,0.25);
}

.metric-value {
    font-size: 34px;
    font-weight: 800;
    color: #FFFFFF;
}

.metric-label {
    font-size: 15px;
    color: #C9C9C9;
    margin-top: 6px;
}

.metric-change {
    font-size: 14px;
    color: #42B21E;
    margin-top: 8px;
    font-weight: 600;
}

.chart-card {
    background-color: #242423;
    padding: 25px;
    border-radius: 18px;
    border: 1px solid #3A3A3A;
    margin-top: 24px;
}

.section-title {
    font-size: 16px;
    letter-spacing: 2px;
    color: #AAA7A0;
    font-weight: 700;
    text-transform: uppercase;
    margin-bottom: 18px;
}

hr {
    border: 1px solid #333333;
}
</style>
""", unsafe_allow_html=True)

# -----------------------------
# Header
# -----------------------------
st.markdown('<div class="main-title">From Grid to Wardrobe</div>', unsafe_allow_html=True)
st.markdown(
    '<div class="subtitle">An interactive FashionTech dashboard analyzing the relationship between Formula 1 team performance and fashion/merch search demand.</div>',
    unsafe_allow_html=True
)

# -----------------------------
# KPIs
# -----------------------------
latest_year = int(analysis_df["year"].max())
latest_df = analysis_df[analysis_df["year"] == latest_year]

top_team = latest_df.sort_values("total_points", ascending=False).iloc[0]
top_fashion_team = latest_df.sort_values("avg_search_interest", ascending=False).iloc[0]

avg_search = analysis_df["avg_search_interest"].mean()
max_spike = analysis_df["max_search_interest"].max()

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-value">{latest_year}</div>
        <div class="metric-label">Latest season analyzed</div>
        <div class="metric-change">↑ recent F1 era</div>
    </div>
    """, unsafe_allow_html=True)

with col2:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-value">{top_team["team_label"]}</div>
        <div class="metric-label">Top performing team</div>
        <div class="metric-change">↑ {top_team["total_points"]:.0f} points</div>
    </div>
    """, unsafe_allow_html=True)

with col3:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-value">{top_fashion_team["team_label"]}</div>
        <div class="metric-label">Highest fashion demand</div>
        <div class="metric-change">↑ avg index {top_fashion_team["avg_search_interest"]:.1f}</div>
    </div>
    """, unsafe_allow_html=True)

with col4:
    st.markdown(f"""
    <div class="metric-card">
        <div class="metric-value">{max_spike:.0f}</div>
        <div class="metric-label">Highest search spike</div>
        <div class="metric-change">Google Trends index</div>
    </div>
    """, unsafe_allow_html=True)

# -----------------------------
# Sidebar Filters
# -----------------------------
st.sidebar.title("Dashboard Filters")

selected_years = st.sidebar.multiselect(
    "Select year(s)",
    sorted(analysis_df["year"].unique()),
    default=sorted(analysis_df["year"].unique())
)

selected_teams = st.sidebar.multiselect(
    "Select team(s)",
    sorted(analysis_df["team_label"].unique()),
    default=sorted(analysis_df["team_label"].unique())
)

filtered_df = analysis_df[
    (analysis_df["year"].isin(selected_years)) &
    (analysis_df["team_label"].isin(selected_teams))
]

filtered_trends = trends_long[
    trends_long["team_label"].isin(selected_teams)
]

# -----------------------------
# Chart 1: Trends Over Time
# -----------------------------
st.markdown('<div class="chart-card">', unsafe_allow_html=True)
st.markdown('<div class="section-title">Fashion Search Interest Over Time</div>', unsafe_allow_html=True)

fig_trends = px.line(
    filtered_trends,
    x="date",
    y="search_interest",
    color="search_term",
    title="",
    labels={
        "date": "Date",
        "search_interest": "Search Interest",
        "search_term": "Search Term"
    }
)

fig_trends.update_traces(line=dict(width=3))

fig_trends.update_layout(
    height=520,
    plot_bgcolor="#242423",
    paper_bgcolor="#242423",
    font=dict(color="#F2F2F2"),
    xaxis=dict(showgrid=True, gridcolor="#333333"),
    yaxis=dict(showgrid=True, gridcolor="#333333", range=[0, 100]),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),
    margin=dict(l=40, r=40, t=50, b=40)
)

st.plotly_chart(fig_trends, use_container_width=True)
st.markdown('</div>', unsafe_allow_html=True)

# -----------------------------
# Chart 2 and 3
# -----------------------------
left, right = st.columns(2)

with left:
    st.markdown('<div class="chart-card">', unsafe_allow_html=True)
    st.markdown('<div class="section-title">Performance vs Fashion Demand</div>', unsafe_allow_html=True)

    fig_scatter = px.scatter(
        filtered_df,
        x="total_points",
        y="avg_search_interest",
        color="team_label",
        size="podiums",
        hover_data=["year", "wins", "avg_finish"],
        labels={
            "total_points": "Total Points",
            "avg_search_interest": "Avg. Search Interest",
            "team_label": "Team"
        }
    )

    fig_scatter.update_traces(
        marker=dict(
            line=dict(width=1, color="#FFFFFF"),
            opacity=0.85
        )
    )

    fig_scatter.update_layout(
        height=430,
        plot_bgcolor="#242423",
        paper_bgcolor="#242423",
        font=dict(color="#F2F2F2"),
        xaxis=dict(showgrid=True, gridcolor="#333333"),
        yaxis=dict(showgrid=True, gridcolor="#333333"),
        margin=dict(l=40, r=40, t=30, b=40)
    )

    st.plotly_chart(fig_scatter, use_container_width=True)
    st.markdown('</div>', unsafe_allow_html=True)

with right:
    st.markdown('<div class="chart-card">', unsafe_allow_html=True)
    st.markdown('<div class="section-title">Team Fashion Demand Ranking</div>', unsafe_allow_html=True)

    ranking_df = filtered_df.groupby("team_label").agg(
        avg_search_interest=("avg_search_interest", "mean")
    ).reset_index().sort_values("avg_search_interest", ascending=True)

    fig_rank = px.bar(
        ranking_df,
        x="avg_search_interest",
        y="team_label",
        orientation="h",
        text="avg_search_interest",
        labels={
            "avg_search_interest": "Avg. Search Interest",
            "team_label": ""
        }
    )

    fig_rank.update_traces(
        marker_color="#2F93FF",
        texttemplate="%{text:.1f}",
        textposition="outside"
    )

    fig_rank.update_layout(
        height=430,
        plot_bgcolor="#242423",
        paper_bgcolor="#242423",
        font=dict(color="#F2F2F2"),
        xaxis=dict(showgrid=True, gridcolor="#333333"),
        yaxis=dict(showgrid=False),
        margin=dict(l=80, r=40, t=30, b=40)
    )

    st.plotly_chart(fig_rank, use_container_width=True)
    st.markdown('</div>', unsafe_allow_html=True)

# -----------------------------
# Chart 4: Performance Ranking
# -----------------------------
st.markdown('<div class="chart-card">', unsafe_allow_html=True)
st.markdown('<div class="section-title">F1 Team Performance Ranking</div>', unsafe_allow_html=True)

performance_rank = filtered_df.groupby("team_label").agg(
    total_points=("total_points", "sum"),
    podiums=("podiums", "sum"),
    wins=("wins", "sum")
).reset_index().sort_values("total_points", ascending=True)

fig_perf = px.bar(
    performance_rank,
    x="total_points",
    y="team_label",
    orientation="h",
    color="wins",
    text="total_points",
    labels={
        "total_points": "Total Points",
        "team_label": "",
        "wins": "Wins"
    },
    color_continuous_scale="Blues"
)

fig_perf.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside"
)

fig_perf.update_layout(
    height=520,
    plot_bgcolor="#242423",
    paper_bgcolor="#242423",
    font=dict(color="#F2F2F2"),
    xaxis=dict(showgrid=True, gridcolor="#333333"),
    yaxis=dict(showgrid=False),
    coloraxis_showscale=False,
    margin=dict(l=100, r=50, t=40, b=40)
)

st.plotly_chart(fig_perf, use_container_width=True)
st.markdown('</div>', unsafe_allow_html=True)

# -----------------------------
# Insights
# -----------------------------
st.markdown('<div class="chart-card">', unsafe_allow_html=True)
st.markdown('<div class="section-title">Key Insights</div>', unsafe_allow_html=True)

corr = analysis_df["total_points"].corr(analysis_df["avg_search_interest"])

st.write(f"""
- The dashboard compares **F1 performance metrics** with **fashion and merchandise search interest**.
- The correlation between total team points and average fashion search interest is **{corr:.2f}**.
- This suggests whether team success appears to move with fashion/merch demand.
- Search demand is treated as a proxy for consumer interest, not direct sales.
""")

st.markdown('</div>', unsafe_allow_html=True)

In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
public_url